# FlashNorm Quality Evaluation

Runs wikitext perplexity + MMLU 5-shot + HellaSwag 0-shot on all three
FlashNorm checkpoints vs their sources. Reproduces the Quality table in
the FlashNorm paper.

## Pre-flight checklist

1. **Runtime > Change runtime type > A100 > Save** before running cells.
2. Wait until the top-right shows `A100` before clicking Run all.
3. Run top-to-bottom.
4. When Part C prints the summary, the five rows are what go into the paper's Table 3.
5. **Runtime > Disconnect and delete runtime** when done, to stop burning compute units.

## Timing and budget on A100

| Cell | Time | Units |
|---|---|---|
| Setup (1-4) | ~3 min | ~0.6 |
| A1 perplexity (135M + 1B) | ~5 min | ~1 |
| B1 perplexity (8B) | ~5-10 min | ~2 |
| B2 MMLU 5-shot (8B) | ~20-30 min | ~6 |
| B3 HellaSwag (8B) | ~10-15 min | ~3 |
| **Total** | **~50-65 min** | **~12-14 units** |

## Design notes

- Each eval runs in a **fresh `lm_eval` CLI subprocess** (isolates GPU and lm-eval state between models; prevents cross-model VRAM leaks).
- Subprocess output streams live into the notebook cell so you see progress.
- On failure, the subprocess output above the error line is the diagnostic.
- Results are parsed from the JSON files that `lm_eval` writes to a temp directory.

## 1. Configuration

In [ ]:
# A100 defaults: include everything
INCLUDE_8B_PPL  = True
INCLUDE_8B_MCQA = True

# Context cap for wikitext perplexity. 2048 is standard.
MAX_LEN = 2048

# Batch size for MCQA tasks on A100 (plenty of VRAM headroom).
MCQA_BATCH = 4

## 2. GPU check

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {name}')
    print(f'Memory: {vram:.1f} GB')
    if 'A100' not in name and 'H100' not in name and 'L4' not in name:
        if vram < 20:
            print(f'NOTE: Defaults assume A100. On {name} ({vram:.1f} GB), set INCLUDE_8B_MCQA=False first.')
else:
    print('WARNING: No GPU detected. Runtime -> Change runtime type -> A100 -> Save')

## 3. Install lm-evaluation-harness

In [ ]:
!pip install -q lm-eval
print('lm-eval installed')

## 4. Subprocess helper

In [ ]:
import subprocess, json, os, glob, tempfile, math, sys, time

def run_eval(model_id, task, batch_size=1, num_fewshot=None, timeout_s=14400, max_length=None):
    '''Run lm_eval in a fresh subprocess; output streams live to the cell.
    Returns the task results dict, or None on failure.'''
    if max_length is None:
        max_length = MAX_LEN
    with tempfile.TemporaryDirectory() as out_dir:
        model_args = f'pretrained={model_id},dtype=float16,max_length={max_length}'
        cmd = [
            'lm_eval',
            '--model', 'hf',
            '--model_args', model_args,
            '--tasks', task,
            '--batch_size', str(batch_size),
            '--output_path', out_dir,
        ]
        if num_fewshot is not None:
            cmd += ['--num_fewshot', str(num_fewshot)]

        print(f'\n[{time.strftime("%H:%M:%S")}] subprocess: lm_eval {task} on {model_id}')
        print(f'    max_length={max_length}  batch_size={batch_size}' + (f'  num_fewshot={num_fewshot}' if num_fewshot else ''))
        sys.stdout.flush()
        t0 = time.time()

        try:
            result = subprocess.run(cmd, timeout=timeout_s)
        except subprocess.TimeoutExpired:
            print(f'\n[TIMEOUT] exceeded {timeout_s}s')
            return None

        dt = time.time() - t0
        print(f'\n[{time.strftime("%H:%M:%S")}] subprocess finished rc={result.returncode} in {dt/60:.1f} min')

        if result.returncode != 0:
            print('[FAILURE] error output is above. Likely OOM, network error, or missing dependency.')
            return None

        for jf in sorted(glob.glob(f'{out_dir}/**/*.json', recursive=True)):
            try:
                with open(jf) as f:
                    data = json.load(f)
                if 'results' in data and task in data['results']:
                    return data['results'][task]
            except Exception:
                continue
        print('[WARNING] results JSON not found under', out_dir)
        return None

def ppl_of(r):
    if r is None: return float('nan')
    return float(r.get('word_perplexity,none') or r.get('word_perplexity') or float('nan'))

def acc_of(r):
    if r is None: return float('nan')
    return float(r.get('acc,none') or r.get('acc') or float('nan'))

# PART A - SmolLM2-135M + Llama-3.2-1B perplexity

## A1. Perplexity (~5 min on A100)

In [ ]:
PAIRS_SAFE = [
    ('HuggingFaceTB/SmolLM2-135M', 'open-machine/SmolLM2-135M-FlashNorm'),
    ('unsloth/Llama-3.2-1B',       'open-machine/Llama-3.2-1B-FlashNorm'),
]

ppl_results = {}
for src, fn in PAIRS_SAFE:
    for label, model_id in [('source', src), ('flashified', fn)]:
        res = run_eval(model_id, 'wikitext', batch_size=1)
        ppl_results[(src, label)] = ppl_of(res)
        print(f'  -> word_ppl = {ppl_results[(src, label)]}')

print('\n' + '=' * 72)
print('PART A RESULTS: wikitext perplexity')
print('=' * 72)
print(f'{"Model":<22} {"Source ppl":>12} {"Flashified ppl":>17} {"Delta":>12}')
print('-' * 72)
for src, fn in PAIRS_SAFE:
    s = ppl_results.get((src, 'source'), float('nan'))
    f = ppl_results.get((src, 'flashified'), float('nan'))
    name = src.split('/')[-1]
    print(f'{name:<22} {s:>12.4f} {f:>17.4f} {f-s:>+12.6f}')
print('=' * 72)

# PART B - Llama-3.1-8B evaluations

## B1. Llama-3.1-8B perplexity (~5-10 min on A100)

In [ ]:
ppl_8b = {}
if INCLUDE_8B_PPL:
    for label, model_id in [('source', 'unsloth/Llama-3.1-8B'),
                            ('flashified', 'open-machine/Llama-3.1-8B-FlashNorm')]:
        res = run_eval(model_id, 'wikitext', batch_size=1)
        ppl_8b[label] = ppl_of(res)
        print(f'  -> word_ppl = {ppl_8b[label]}')

    print('\n' + '=' * 72)
    print('B1 RESULT: Llama-3.1-8B wikitext perplexity')
    print('=' * 72)
    s = ppl_8b.get('source', float('nan'))
    f = ppl_8b.get('flashified', float('nan'))
    print(f'{"Llama-3.1-8B":<22} source={s:.4f}  flashified={f:.4f}  delta={f-s:+.6f}')
    print('=' * 72)
else:
    print('B1 skipped (INCLUDE_8B_PPL=False).')

## B2. Llama-3.1-8B MMLU 5-shot (~20-30 min on A100)

In [ ]:
mmlu = {}
if INCLUDE_8B_MCQA:
    for label, model_id in [('source', 'unsloth/Llama-3.1-8B'),
                            ('flashified', 'open-machine/Llama-3.1-8B-FlashNorm')]:
        res = run_eval(model_id, 'mmlu', batch_size=MCQA_BATCH, num_fewshot=5, max_length=4096)
        mmlu[label] = acc_of(res)
        print(f'  -> MMLU acc = {mmlu[label]}')

    print('\n' + '=' * 72)
    print('B2 RESULT: Llama-3.1-8B MMLU 5-shot')
    print('=' * 72)
    s = mmlu.get('source', float('nan'))
    f = mmlu.get('flashified', float('nan'))
    print(f'{"Llama-3.1-8B":<22} source={s:.4f}  flashified={f:.4f}  delta={f-s:+.4f}')
    print('=' * 72)
else:
    print('B2 skipped (INCLUDE_8B_MCQA=False).')

## B3. Llama-3.1-8B HellaSwag 0-shot (~10-15 min on A100)

In [ ]:
hs = {}
if INCLUDE_8B_MCQA:
    for label, model_id in [('source', 'unsloth/Llama-3.1-8B'),
                            ('flashified', 'open-machine/Llama-3.1-8B-FlashNorm')]:
        res = run_eval(model_id, 'hellaswag', batch_size=MCQA_BATCH, max_length=2048)
        hs[label] = acc_of(res)
        print(f'  -> HellaSwag acc = {hs[label]}')

    print('\n' + '=' * 72)
    print('B3 RESULT: Llama-3.1-8B HellaSwag zero-shot')
    print('=' * 72)
    s = hs.get('source', float('nan'))
    f = hs.get('flashified', float('nan'))
    print(f'{"Llama-3.1-8B":<22} source={s:.4f}  flashified={f:.4f}  delta={f-s:+.4f}')
    print('=' * 72)
else:
    print('B3 skipped (INCLUDE_8B_MCQA=False).')

# PART C - Summary

The block below reproduces the rows of the paper's Quality table.

In [ ]:
def fmt(v, prec=4):
    return 'nan' if (isinstance(v, float) and math.isnan(v)) else f'{v:.{prec}f}'
def fmt_d(v, prec=6):
    return 'nan' if (isinstance(v, float) and math.isnan(v)) else f'{v:+.{prec}f}'

print('=' * 80)
print('QUALITY TABLE SUMMARY')
print('=' * 80)

for src, fn in PAIRS_SAFE:
    s = ppl_results.get((src, 'source'), float('nan'))
    f = ppl_results.get((src, 'flashified'), float('nan'))
    name = src.split('/')[-1]
    print(f'wikitext word_ppl   {name:<25} source={fmt(s)}  flashified={fmt(f)}  delta={fmt_d(f-s)}')

if INCLUDE_8B_PPL:
    s = ppl_8b.get('source', float('nan'))
    f = ppl_8b.get('flashified', float('nan'))
    print(f'wikitext word_ppl   {"Llama-3.1-8B":<25} source={fmt(s)}  flashified={fmt(f)}  delta={fmt_d(f-s)}')

if INCLUDE_8B_MCQA:
    s = mmlu.get('source', float('nan'))
    f = mmlu.get('flashified', float('nan'))
    print(f'MMLU 5-shot         {"Llama-3.1-8B":<25} source={fmt(s)}  flashified={fmt(f)}  delta={fmt_d(f-s, 4)}')
    s = hs.get('source', float('nan'))
    f = hs.get('flashified', float('nan'))
    print(f'HellaSwag 0-shot    {"Llama-3.1-8B":<25} source={fmt(s)}  flashified={fmt(f)}  delta={fmt_d(f-s, 4)}')

print()
print(f'(INCLUDE_8B_PPL={INCLUDE_8B_PPL}, INCLUDE_8B_MCQA={INCLUDE_8B_MCQA}, MAX_LEN={MAX_LEN}, MCQA_BATCH={MCQA_BATCH})')
print('=' * 80)

## Optional: auto-disconnect runtime when done

A connected A100 runtime burns ~12 compute units per hour whether cells
are running or idle. Run this cell to terminate the runtime once the
summary has printed. The cell output above persists across disconnects,
so the Quality table remains readable.

In [ ]:
import time
print('\n[auto-disconnect] summary printed. Disconnecting in 90 seconds.')
print('[auto-disconnect] press the stop button to cancel if you want to keep the runtime.')
for i in range(9, 0, -1):
    time.sleep(10)
    print(f'[auto-disconnect] {i * 10}s remaining...')
print('[auto-disconnect] disconnecting now.')
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f'[auto-disconnect] failed: {e}')
    print('[auto-disconnect] manually: Runtime > Disconnect and delete runtime')